In [1]:
from sage.all import *

def solve_y_step_by_step(G_poly, t_var, precision):
    K = G_poly.base_ring()
    c1 = G_poly[1] 
    if c1 == 0:
        raise ValueError("The coefficient of the linear term of G(y) cannot be 0 (it is not invertible and its inverse function cannot be found).")
    c1_inv = c1.inverse()
    
    y_poly = c1_inv * t_var
    
    for k in range(2, precision):
        current_val = G_poly(y_poly) 
        residue = current_val[k] 
        if residue == 0:
            continue
        b_k = -residue * c1_inv
        y_poly += b_k * (t_var ** k)
        
    return y_poly


def local_expansion(f_poly, M_poly, precision=5):

    F2 = GF(2)  
    R = PolynomialRing(F2, 'x')  
    f = R(f_poly) 
    M = R(M_poly) 

    d = M.degree()
    K = GF(2**d, name='alpha', modulus=M)
    alpha = K.gen()  
    R_t = PolynomialRing(K, 't')
    t = R_t.gen()

    R_y = PolynomialRing(K, 'y')
    y_var = R_y.gen()
    G_poly_y = M(alpha + y_var) 
    y_poly_t = solve_y_step_by_step(G_poly_y, t, precision + 2)
    
    x_poly_t = alpha + y_poly_t
    F_poly_t = f(x_poly_t)
    S = PowerSeriesRing(K, 't', default_prec=precision)
    return S(F_poly_t).add_bigoh(precision)

In [2]:
def get_expansion_vectors(f_poly, M_poly, num_terms=3):
    d = M_poly.degree() 
    result_series = local_expansion(f_poly, M_poly, precision=num_terms)
    coeffs = result_series.list() 
    
    vectors = []
    for i in range(num_terms):
        if i < len(coeffs):
            c = coeffs[i] 
            vec = c.polynomial().list()  
            vec = [int(v) for v in vec] 
            vec += [0] * (d - len(vec)) 
        else:
            vec = [0] * d  
        vectors.append(vec)
    
    return vectors


def process_file(input_path, output_path=None):
   
    F2 = GF(2)
    R = PolynomialRing(F2, 'x')
    x = R.gen()
    M = x^2 + x + 1
    
    results = []
    
    with open(input_path, 'r') as file:
        for line_num, line in enumerate(file, 1):
            line = line.strip()
            if not line:
                continue
            
            try:
                f = R(line)  #
                vectors = get_expansion_vectors(f, M, num_terms=3)
                results.append(vectors)
            except Exception as e:
                print(f"Line {line_num} processing failed.: {e}")
    if output_path:
        with open(output_path, 'w') as out:
            for vec in results:
                out.write(str(vec) + '\n')
    
    return results

input_file = "mul_line/classcial_mul/gf233/2^233_l(2D)_pn_basis.txt"
output_file = "/mul_line/classcial_mul/gf233/expansion_results.txt"

print("=== Batch calculation of partial expansion ===")
print(f"Input file: {input_file}")
print(f"Irreducible polynomial: x^2 + x + 1 (degree 2)")
print(f"Number of terms expanded: 3")
results = process_file(input_file, output_file)

print(f"\nA total of {len(results)} polynomials were processed")
print(f"Results have been saved to: {output_file}")

=== Batch calculation of partial expansion ===
Input file: mul_line/classcial_mul/gf233/2^233_l(2D)_pn_basis.txt
Irreducible polynomial: x^2 + x + 1 (degree 2)
Number of terms expanded: 3


FileNotFoundError: [Errno 2] No such file or directory: 'mul_line/classcial_mul/gf233/2^233_l(2D)_pn_basis.txt'